In [1]:
# ==============================================================================
# Análisis Bivariado: Categórico vs Categórico
# Autor: Asistente IA para Matias Zapata
# Fecha: 2026-06-08
# ==============================================================================

# 1. Importación de Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings

# Configuración de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings('ignore')

In [2]:
# 2. Carga de Datos
# Asegúrate de que el archivo 'train_processed.csv' esté en el mismo directorio
try:
    df = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)
    print("✅ Datos cargados correctamente.")
    print(f"Dimensiones del dataset: {df.shape}")
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'train_processed.csv'.")

✅ Datos cargados correctamente.
Dimensiones del dataset: (11840, 46)


In [3]:
# 3. Preprocesamiento Inicial
# Identificamos columnas categóricas típicas en este dataset de viviendas
# Basado en la estructura usual de estos datos (Miami Housing Dataset):
categorical_cols = ['property_type', 'zipcode', 'state', 'waterfront', 'view', 'condition']

# Filtramos solo las columnas que existen en el dataframe actual
existing_cat_cols = [col for col in categorical_cols if col in df.columns]

# Si hay columnas numéricas que representan categorías (ej. 0/1), las convertimos
if 'waterfront' in df.columns:
    df['waterfront'] = df['waterfront'].astype('category')
if 'view' in df.columns:
    df['view'] = df['view'].astype('category')

print(f"\n📊 Variables categóricas identificadas para el análisis: {existing_cat_cols}")


📊 Variables categóricas identificadas para el análisis: ['zipcode']


In [4]:
# 4. Función para Análisis Bivariado Categórico vs Categórico
def analyze_categorical_vs_categorical(df, col1, col2, alpha=0.05):
    """
    Realiza un análisis bivariado entre dos variables categóricas.
    Incluye: Tabla de contingencia, Gráfico de barras agrupadas y Prueba Chi-Cuadrado.
    """
    
    # Crear tabla de contingencia
    contingency_table = pd.crosstab(df[col1], df[col2])
    
    # Verificar si la tabla tiene suficientes datos para la prueba
    if contingency_table.empty or contingency_table.sum().sum() == 0:
        print(f"⚠️ No hay datos suficientes para analizar {col1} vs {col2}.")
        return

    # --- Prueba Estadística: Chi-Cuadrado de Independencia ---
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    print(f"\n{'='*60}")
    print(f"🔍 Análisis: {col1.upper()} vs {col2.upper()}")
    print(f"{'='*60}")
    print(f"📋 Tabla de Contingencia (Muestras):\n{contingency_table.head()}")
    print(f"\n📈 Resultados Prueba Chi-Cuadrado:")
    print(f"   - Estadístico Chi2: {chi2:.4f}")
    print(f"   - Valor p (p-value): {p_value:.4e}")
    print(f"   - Grados de libertad: {dof}")
    
    if p_value < alpha:
        print(f"   ✅ Conclusión: Existe una asociación estadísticamente significativa (p < {alpha}).")
    else:
        print(f"   ❌ Conclusión: No hay evidencia suficiente de asociación (p >= {alpha}).")

    # --- Visualización: Gráfico de Barras Agrupadas ---
    # Normalizamos por fila para ver porcentajes dentro de cada categoría de col1
    contingency_pct = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100
    
    plt.figure(figsize=(14, 7))
    ax = sns.barplot(data=contingency_pct.reset_index().melt(id_vars=[col1]), 
                     x=col1, y='value', hue=col2)
    
    plt.title(f'Distribución de {col2} por cada {col1} (%)', fontsize=14)
    plt.xlabel(col1)
    plt.ylabel('Porcentaje (%)')
    plt.legend(title=col2, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [5]:
# 5. Ejecución del Análisis
# Seleccionamos pares de variables interesantes para analizar
# Puedes modificar esta lista según las columnas presentes en tu CSV
pairs_to_analyze = [
    ('property_type', 'zipcode'),
    ('property_type', 'waterfront'),
    ('zipcode', 'view')
]

# Filtramos pares válidos
valid_pairs = [(c1, c2) for c1, c2 in pairs_to_analyze if c1 in df.columns and c2 in df.columns]

if valid_pairs:
    for col1, col2 in valid_pairs:
        analyze_categorical_vs_categorical(df, col1, col2)
else:
    print("⚠️ No se encontraron pares de variables categóricas válidas para analizar.")

# ==============================================================================
# EXTRA: Si deseas comparar Categórica vs Numérica (ANOVA/T-test)
# ==============================================================================
# Descomenta el siguiente bloque si quieres ver diferencias de PRECIO por TIPO de propiedad

"""
def analyze_categorical_vs_numerical(df, cat_col, num_col, alpha=0.05):
    groups = [group[num_col].dropna() for name, group in df.groupby(cat_col)]
    
    # Si hay más de 2 grupos, usamos ANOVA; si son 2, T-test
    if len(groups) > 2:
        f_stat, p_value = stats.f_oneway(*groups)
        test_name = "ANOVA"
    elif len(groups) == 2:
        t_stat, p_value = stats.ttest_ind(*groups, equal_var=False)
        test_name = "T-Test"
    else:
        return
        
    print(f"\n🧪 Prueba {test_name}: {num_col} por {cat_col}")
    print(f"   - Valor p: {p_value:.4e}")
    if p_value < alpha:
        print(f"   ✅ Hay diferencias significativas en '{num_col}' entre los grupos de '{cat_col}'.")
    
    # Boxplot
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=cat_col, y=num_col, data=df)
    plt.title(f'{num_col} por {cat_col}')
    plt.xticks(rotation=45)
    plt.show()

# Ejemplo de uso:
# from scipy import stats
# analyze_categorical_vs_numerical(df, 'property_type', 'price')
"""

⚠️ No se encontraron pares de variables categóricas válidas para analizar.


'\ndef analyze_categorical_vs_numerical(df, cat_col, num_col, alpha=0.05):\n    groups = [group[num_col].dropna() for name, group in df.groupby(cat_col)]\n\n    # Si hay más de 2 grupos, usamos ANOVA; si son 2, T-test\n    if len(groups) > 2:\n        f_stat, p_value = stats.f_oneway(*groups)\n        test_name = "ANOVA"\n    elif len(groups) == 2:\n        t_stat, p_value = stats.ttest_ind(*groups, equal_var=False)\n        test_name = "T-Test"\n    else:\n        return\n\n    print(f"\n🧪 Prueba {test_name}: {num_col} por {cat_col}")\n    print(f"   - Valor p: {p_value:.4e}")\n    if p_value < alpha:\n        print(f"   ✅ Hay diferencias significativas en \'{num_col}\' entre los grupos de \'{cat_col}\'.")\n\n    # Boxplot\n    plt.figure(figsize=(12, 6))\n    sns.boxplot(x=cat_col, y=num_col, data=df)\n    plt.title(f\'{num_col} por {cat_col}\')\n    plt.xticks(rotation=45)\n    plt.show()\n\n# Ejemplo de uso:\n# from scipy import stats\n# analyze_categorical_vs_numerical(df, \'prope